In [1]:
import os

In [2]:
os.chdir('..')

In [3]:
%pwd

'd:\\Chicken Disease Classification'

In [4]:
import tensorflow as tf

In [5]:
from dataclasses import dataclass
from pathlib import Path

In [6]:
@dataclass
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int
    

In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories,save_json

In [9]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_validation_config(self)->EvaluationConfig:
        eval_config=EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/Chicken-fecal-images",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [22]:
class Evaluation:
    def __init__(self,config: EvaluationConfig):
        self.config= config
    
    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )
        
    @staticmethod
    def load_model(path: Path):
        return tf.keras.models.load_model(path)
    
    def evaluation(self):
        self.model=self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score=self.model.evaluate(self.valid_generator)
        print(self.score)
    
    def save_score(self):
        scores = {"loss": float(self.score[0]), "accuracy": float(self.score[1])}
        save_json(path=Path("scores.json"), data=scores)

In [23]:
try:
    config= ConfigurationManager()
    val_config=config.get_validation_config()
    evaluation=Evaluation(val_config)
    evaluation.evaluation()
    
    
except Exception as e:
    raise e

[2026-03-25 20:45:37,837: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-25 20:45:37,845: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-25 20:45:37,853: INFO: common: created directory at: artifacts]
[2026-03-25 20:45:38,570: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 116 images belonging to 2 classes.


e:\Miniconda3\envs\Chicken\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


8/8 ━━━━━━━━━━━━━━━━━━━━ 53s 6s/step - accuracy: 0.9395 - loss: 0.1721
[0.18069665133953094, 0.9482758641242981]


In [24]:
evaluation.save_score()

[2026-03-25 20:46:42,567: INFO: common: json file saved at: scores.json]
